# Calculation of Several Operational Performance Metrics Based on Carris Metropolitana Demand and Supply

Global values and grouped by operator


Example of metrics :

Percentage of planned 'trips' that were executed

Median observed speed

Percentage of executed 'trips' that started on time

Total kilometers executed in service

Median travel time of a 'trip'

Ratio of passengers per executed kilometer in service



## Prepare environment

In [ ]:
# =========================
# FILTERS FOR DATA EXTRACTION
# =========================
START_DATE = "20260101"
END_DATE   = "20260228"

# =========================
# VARS FOR PARALLEL PROCESSING
# =========================
MAX_WORKERS = 30
CHUNK_SIZE_DAYS = 15

# =========================
# GLOBAL VARS TO GET FROM CONFIG FILE
# =========================

required_vars = [
    "MONGO_URI",
    "DB_NAME",
    "COLLECTION_NAME", #rideones
    "TIMEZONE",
    "OUTPUT_FILES_FOLDER",
]



In [ ]:
# from config py file

import importlib
import config

# import, cleaning the cache to get latest changes
importlib.reload(config)


print("Config file in:", config.__file__)


missing = []
locals_dict = locals()

for name in required_vars:
    if hasattr(config, name):
        locals_dict[name] = getattr(config, name)
    else:
        missing.append(name)

if missing:
    raise RuntimeError(f"Missing config variables: {missing}")

# Create directory for output files if it doesn't exist
import os
from pathlib import Path
OUTPUT_FILES_FOLDER.mkdir(parents=True, exist_ok=True)

[name for name in required_vars if hasattr(config, name) and print(name, getattr(config, name))]

## Get rides

In [ ]:
# =========================
# Imports
# =========================
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from pymongo import MongoClient
import pandas as pd

# =========================
# MongoDB connection (GLOBAL)
# =========================
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# =========================
# Generate date chunks
# =========================
def generate_chunks(start, end, size):
    """
    Generates inclusive date chunks with no overlap and no gaps.
    START_DATE and END_DATE are both included.
    """
    start_dt = datetime.strptime(start, "%Y%m%d")
    end_dt = datetime.strptime(end, "%Y%m%d")

    chunks = []
    cur = start_dt
    while cur <= end_dt:
        nxt = min(cur + timedelta(days=size - 1), end_dt)
        chunks.append({
            "start": cur.strftime("%Y%m%d"),
            "end": nxt.strftime("%Y%m%d")
        })
        cur = nxt + timedelta(days=1)
    return chunks

date_chunks = generate_chunks(START_DATE, END_DATE, CHUNK_SIZE_DAYS)
print(f"Chunks: {len(date_chunks)}")

# =========================
# Fetch per chunk (FIELDS ONLY)
# =========================
def fetch_chunk(chunk):
    pipeline = [
        {
            "$match": {
                "operational_date": {
                    "$gte": chunk["start"],
                    "$lte": chunk["end"]  # inclusive, matches chunk logic
                },
                "agency_id": {"$in": ["41", "42", "43", "44"]}
            }
        },
        {
            "$project": {
                "_id": 1,
                "agency_id": 1,
                "line_id": 1,
                "route_id": 1,
                "trip_id": 1,
                "three_vehicle_events_grade": "$analysis.SIMPLE_THREE_VEHICLE_EVENTS.grade",
                "expected_start_time_grade": "$analysis.EXPECTED_START_TIME.grade",
                "end_time_observed": 1,
                "extension_observed": 1,
                "extension_scheduled": 1,
                "operational_date": 1,
                "start_time_observed": 1,
                "start_time_scheduled": 1,
                "end_time_scheduled": 1,
                "passengers_observed": 1,
            }
        }
    ]

    return list(collection.aggregate(pipeline, allowDiskUse=True))

# =========================
# Parallel execution
# =========================
rows = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_chunk, c) for c in date_chunks]

    for i, f in enumerate(as_completed(futures), 1):
        chunk_rows = f.result()
        rows.extend(chunk_rows)
        print(f"Processed {i}/{len(date_chunks)} chunks | rows so far: {len(rows)}")

# =========================
# Final DataFrame
# =========================
rides_df2 = pd.DataFrame(rows)


In [ ]:
rides_df2.head(3)

In [ ]:
#save to CSV

rides_df2.to_csv(os.path.join(OUTPUT_FILES_FOLDER, "rides_performance_2025_raw.csv"),
     index=False
 )


In [ ]:
import pandas as pd

rides_df2= pd.read_csv(os.path.join(OUTPUT_FILES_FOLDER, "rides_performance_2025_raw.csv")) 

## Calculate rides metrics

In [ ]:
import numpy as np
import pandas as pd

# =========================
# Filter only PASS
# =========================
rides_start_time_dif = rides_df2.copy()

# =========================
# Ensure numeric timestamps
# =========================
rides_start_time_dif["start_time_observed"] = pd.to_numeric(
    rides_start_time_dif["start_time_observed"], errors="coerce"
)

rides_start_time_dif["start_time_scheduled"] = pd.to_numeric(
    rides_start_time_dif["start_time_scheduled"], errors="coerce"
)

# =========================
# Compute difference (minutes)
# observed - scheduled
# =========================
rides_start_time_dif["start_time_dif_observedscheduled"] = ((
    rides_start_time_dif["start_time_observed"]
    - rides_start_time_dif["start_time_scheduled"]
) / (1000 * 60)).round(2)


# =========================
# Compute difference (meters)
# observed - scheduled
# =========================
rides_start_time_dif["extension_dif_observedscheduled"] = (
    rides_start_time_dif["extension_observed"]
    - rides_start_time_dif["extension_scheduled"]
).round(2)

# =========================
# Get delays status
# =========================
rides_start_time_dif["delay_over5min"] = np.where(
    rides_start_time_dif["start_time_dif_observedscheduled"] > 5,
    1,
    0
)

rides_start_time_dif["delay_over10min"] = np.where(
    rides_start_time_dif["start_time_dif_observedscheduled"] > 10,
    1,
    0
)

# =========================
# Compute speed
# =========================
rides_start_time_dif["speed_observed"] = np.where(
    (rides_start_time_dif["end_time_observed"] > rides_start_time_dif["start_time_observed"]) &
    (rides_start_time_dif["extension_observed"] > 0),
    (rides_start_time_dif["extension_observed"] / 1000) /
    ((rides_start_time_dif["end_time_observed"] - rides_start_time_dif["start_time_observed"]) / 3_600_000),
    np.nan
)

rides_start_time_dif["speed_scheduled"] = np.where(
    (rides_start_time_dif["end_time_scheduled"] > rides_start_time_dif["start_time_scheduled"]) &
    (rides_start_time_dif["extension_scheduled"] > 0),
    (rides_start_time_dif["extension_scheduled"] / 1000) /
    ((rides_start_time_dif["end_time_scheduled"] - rides_start_time_dif["start_time_scheduled"]) / 3_600_000),
    np.nan
)

# =========================
# Compute execution categories
# =========================

rides_start_time_dif["three_events_pass_or_ticketing"] = np.where((
    rides_start_time_dif["three_vehicle_events_grade"] == "pass") | 
    (rides_start_time_dif["passengers_observed"] > 0)
    , True, False)

rides_start_time_dif["three_events_fail_with_ticketing"] = np.where((
    rides_start_time_dif["three_vehicle_events_grade"] == "fail") & 
    (rides_start_time_dif["passengers_observed"] > 0)
    , True, False)

rides_start_time_dif["ticketing"] = np.where( 
    (rides_start_time_dif["passengers_observed"] > 0)
    , True, False)

rides_start_time_dif["three_events_pass_with_no_ticketing"] = np.where((
    rides_start_time_dif["three_vehicle_events_grade"] == "pass") & 
    (rides_start_time_dif["passengers_observed"] < 1)
    , True, False)


# =========================
# Compute time durations
# =========================
rides_start_time_dif["time_duration_observed"] = rides_start_time_dif["end_time_observed"] - rides_start_time_dif["start_time_observed"]

rides_start_time_dif["time_duration_observed_min"] = (rides_start_time_dif["end_time_observed"] - rides_start_time_dif["start_time_observed"]) / 60000

rides_start_time_dif["time_duration_scheduled"] = rides_start_time_dif["end_time_scheduled"] - rides_start_time_dif["start_time_scheduled"]

rides_start_time_dif["time_duration_scheduled_min"] = (rides_start_time_dif["end_time_scheduled"] - rides_start_time_dif["start_time_scheduled"]) / 60000

rides_start_time_dif["time_duration_dif_observedscheduled"] = rides_start_time_dif["time_duration_observed"] - rides_start_time_dif["time_duration_scheduled"]

rides_start_time_dif["time_duration_dif_observedscheduled_min"] = rides_start_time_dif["time_duration_observed_min"] - rides_start_time_dif["time_duration_scheduled_min"]



# =========================
# Compute start time grade combinations
# =========================

rides_start_time_dif["expected_start_time_pass_for_three_events_pass_or_ticketing"] = np.where(
    (rides_start_time_dif["expected_start_time_grade"] == "pass") & 
    (rides_start_time_dif["three_events_pass_or_ticketing"] == True)
    , True, False)

In [ ]:
rides_start_time_dif["extension_observed_km"] = (rides_start_time_dif["extension_observed"] / 1000)

rides_start_time_dif["time_duration_observed_min"] = (rides_start_time_dif["time_duration_observed"] / 1000 / 60)

rides_start_time_dif["year_month"] = rides_start_time_dif["operational_date"].astype(str).str.slice(0, 6)#.str.slice(0,6)    

In [ ]:
rides_start_time_dif.head()

In [ ]:
#save to CSV

rides_start_time_dif.to_csv(os.path.join(OUTPUT_FILES_FOLDER, "rides_performance_2025_detailed.csv"),
     index=False
 )


## Aggregated metrics by Agency

In [ ]:
import pandas as pd
rides_start_time_dif = pd.read_csv(INPUT_FILES_FOLDER/"rides_performance_2025_detailed.csv", encoding="utf-8",sep=",")

In [ ]:
len(rides_start_time_dif)

In [ ]:
rides_start_time_dif.columns

In [ ]:
# FILTER FOR THE MONTH IN ANALYSIS IF NEEDED


rides_start_time_dif_month = rides_start_time_dif[
    (rides_start_time_dif['operational_date'] >= 20260101) &
    (rides_start_time_dif['operational_date'] <= 20260131)
].copy()

In [ ]:
import pandas as pd
import numpy as np


df_summary_area = (
    rides_start_time_dif_month #rides_start_time_dif        ##### change dataset to monthly or total
    .groupby("agency_id")
    .agg(
        operational_date=("operational_date", "nunique"),
        passengers_observed=("passengers_observed", "sum"),
        total_trips=("_id", "count"),
        
        three_vehicle_events_pass=("three_vehicle_events_grade", lambda x: (x == "pass").sum()),
        three_vehicle_events_fail=("three_vehicle_events_grade", lambda x: (x == "fail").sum()),
        three_events_pass_or_ticketing=("three_events_pass_or_ticketing", lambda x: (x == True).sum()),
        three_events_fail_with_ticketing=("three_events_fail_with_ticketing", lambda x: (x == True).sum()),
        three_events_pass_with_no_ticketing=("three_events_pass_with_no_ticketing", lambda x: (x == True).sum()),
        ticketing=("ticketing", lambda x: (x == True).sum()),

        extension_observed_km=("extension_observed", lambda x: x.sum() / 1000),
        extension_scheduled_km=("extension_scheduled", lambda x: x.sum() / 1000),

        extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km=("extension_scheduled", lambda x: x[rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"]].sum() / 1000),
        extension_observed_for_three_vehicle_events_pass_or_ticketing_km=("extension_observed", lambda x: x[rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"]].sum() / 1000),
        median_extension_dif_observedscheduled_for_three_vehicle_events_pass_or_ticketing_km=("extension_dif_observedscheduled", lambda x: x[rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"]].median() / 1000),

        time_duration_observed_for_three_vehicle_events_pass_or_ticketing_h=("time_duration_observed", lambda x: x[rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"]].sum() / 1000 / 3600),
        median_time_duration_observed_for_three_vehicle_events_pass_or_ticketing_min=("time_duration_observed_min", lambda x: x[rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"]].median()),

        median_speed_observed_for_three_vehicle_events_pass_or_ticketing_kmh=("speed_observed", lambda x: x[rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"]].median()),
        median_speed_scheduled_for_three_vehicle_events_pass_or_ticketing_kmh=("speed_scheduled", lambda x: x[rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"]].median()),



        expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing=("expected_start_time_grade", lambda x: (rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"] & (x == "pass")).sum()),
        expected_start_time_fail_for_three_vehicle_events_pass_or_ticketing=("expected_start_time_grade", lambda x: (rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"] & (x == "fail")).sum()),

        start_time_below5min_for_three_vehicle_events_pass_or_ticketing=("delay_over5min", lambda x: (rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"] & (x == 0)).sum()),
        start_time_delay_over5min_for_three_vehicle_events_pass_or_ticketing=("delay_over5min", lambda x: x[rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"]].sum()),

        start_time_below10min_for_three_vehicle_events_pass_or_ticketing=("delay_over10min", lambda x: (rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"] & (x == 0)).sum()),
        start_time_delay_over10min_for_three_vehicle_events_pass_or_ticketing=("delay_over10min", lambda x: x[rides_start_time_dif.loc[x.index, "three_events_pass_or_ticketing"]].sum()),
     

        
    )
    .reset_index()
)

df_summary_area

In [ ]:
df_summary_area.columns

In [ ]:
df_summary_area["dif_extension_observedscheduled_for_three_vehicle_events_pass_or_ticketing_km"] = df_summary_area["extension_observed_for_three_vehicle_events_pass_or_ticketing_km"] - df_summary_area["extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km"]
df_summary_area["dif_median_speed_observedscheduled_for_three_vehicle_events_pass_or_ticketing_kmh"] = df_summary_area["median_speed_observed_for_three_vehicle_events_pass_or_ticketing_kmh"] - df_summary_area["median_speed_scheduled_for_three_vehicle_events_pass_or_ticketing_kmh"]

df_summary_area["pct_three_events_pass_or_ticketing_from_total_trips"] = df_summary_area['three_events_pass_or_ticketing'] / df_summary_area['total_trips'] * 100
df_summary_area["pct_three_events_pass_or_ticketing_oposite_from_total_trips"] = (df_summary_area['total_trips'] - df_summary_area['three_events_pass_or_ticketing']) / df_summary_area['total_trips'] * 100
df_summary_area["pct_expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing_from_three_events_pass_or_ticketing"] = df_summary_area['expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing'] / df_summary_area['three_events_pass_or_ticketing'] * 100
df_summary_area["pct_start_time_below5min_for_three_vehicle_events_pass_or_ticketing_from_three_events_pass_or_ticketing"] = df_summary_area['start_time_below5min_for_three_vehicle_events_pass_or_ticketing'] / df_summary_area['three_events_pass_or_ticketing'] * 100
df_summary_area["pct_expected_start_time_fail_or_na_from_three_events_pass_or_ticketing"] = (df_summary_area['three_events_pass_or_ticketing'] - df_summary_area['expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing']) / df_summary_area['three_events_pass_or_ticketing'] * 100
df_summary_area["pct_three_events_pass_with_no_ticketing_from_three_events_pass_or_ticketing"] = df_summary_area["three_events_pass_with_no_ticketing"] / df_summary_area['three_events_pass_or_ticketing'] * 100

df_summary_area['average_total_trips_per_day'] = df_summary_area['total_trips'] / df_summary_area['operational_date']
df_summary_area['average_three_events_pass_or_ticketing_per_day'] = df_summary_area['three_events_pass_or_ticketing'] / df_summary_area['operational_date']
df_summary_area['average_expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing_per_day'] = df_summary_area['expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing'] / df_summary_area['operational_date']

df_summary_area['average_start_time_below5min_for_three_vehicle_events_pass_or_ticketing_per_day'] = df_summary_area['start_time_below5min_for_three_vehicle_events_pass_or_ticketing'] / df_summary_area['operational_date']
df_summary_area['average_extension_observed_for_three_vehicle_events_pass_or_ticketing_km_per_day'] = df_summary_area['extension_observed_for_three_vehicle_events_pass_or_ticketing_km'] / df_summary_area['operational_date']
df_summary_area['average_extension_observed_for_three_vehicle_events_pass_or_ticketing_km_per_three_events_pass_or_ticketing'] = df_summary_area['extension_observed_for_three_vehicle_events_pass_or_ticketing_km'] / df_summary_area['three_events_pass_or_ticketing']

df_summary_area["dif_three_events_pass_or_ticketing_from_total_trips"] = df_summary_area["total_trips"] - df_summary_area["three_events_pass_or_ticketing"]


df_summary_area['average_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km_per_day'] = df_summary_area['extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km'] / df_summary_area['operational_date']
df_summary_area['average_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km_per_three_events_pass_or_ticketing'] = df_summary_area['extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km'] / df_summary_area['three_events_pass_or_ticketing']

df_summary_area["dif_extension_scheduled_extension_sheduled_of_three_vehicle_events_pass_or_ticketing_km"] = df_summary_area["extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km"] -df_summary_area["extension_scheduled_km"]



In [ ]:
df_summary_area["pct_three_events_pass_or_ticketing_from_total_trips"] = df_summary_area['three_events_pass_or_ticketing'] / df_summary_area['total_trips'] * 100
df_summary_area["passengers_observed_per_three_events_pass_or_ticketing"] = df_summary_area['passengers_observed'] / df_summary_area['three_events_pass_or_ticketing']
df_summary_area["passengers_observed_per_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km"] = df_summary_area['passengers_observed'] / df_summary_area['extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km']

#### Add AML metrics (global metrics values)

In [ ]:
aml_row = {
    "agency_id": "AML",
    "operational_date": df_summary_area["operational_date"].mean(),
    "passengers_observed": df_summary_area["passengers_observed"].sum(),
    "total_trips": df_summary_area["total_trips"].sum(),
    "three_vehicle_events_pass": df_summary_area["three_vehicle_events_pass"].sum(),
    "three_vehicle_events_fail": df_summary_area["three_vehicle_events_fail"].sum(),
    "three_events_pass_or_ticketing": df_summary_area["three_events_pass_or_ticketing"].sum(),
    "three_events_fail_with_ticketing": df_summary_area["three_events_fail_with_ticketing"].sum(),
    "three_events_pass_with_no_ticketing": df_summary_area["three_events_pass_with_no_ticketing"].sum(),
    "ticketing": df_summary_area["ticketing"].sum(),
    "extension_observed_km": df_summary_area["extension_observed_km"].sum(),
    "extension_scheduled_km": df_summary_area["extension_scheduled_km"].sum(),
    "extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km": df_summary_area["extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km"].sum(),
    "extension_observed_for_three_vehicle_events_pass_or_ticketing_km": df_summary_area["extension_observed_for_three_vehicle_events_pass_or_ticketing_km"].sum(),
    "median_extension_dif_observedscheduled_for_three_vehicle_events_pass_or_ticketing_km": df_summary_area["median_extension_dif_observedscheduled_for_three_vehicle_events_pass_or_ticketing_km"].median(),
    "time_duration_observed_for_three_vehicle_events_pass_or_ticketing_h": df_summary_area["time_duration_observed_for_three_vehicle_events_pass_or_ticketing_h"].sum(),
    "median_time_duration_observed_for_three_vehicle_events_pass_or_ticketing_min": df_summary_area["median_time_duration_observed_for_three_vehicle_events_pass_or_ticketing_min"].median(),
    "median_speed_observed_for_three_vehicle_events_pass_or_ticketing_kmh": df_summary_area["median_speed_observed_for_three_vehicle_events_pass_or_ticketing_kmh"].median(),
    "median_speed_scheduled_for_three_vehicle_events_pass_or_ticketing_kmh": df_summary_area["median_speed_scheduled_for_three_vehicle_events_pass_or_ticketing_kmh"].median(),
    "expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing": df_summary_area["expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing"].sum(),
    "expected_start_time_fail_for_three_vehicle_events_pass_or_ticketing": df_summary_area["expected_start_time_fail_for_three_vehicle_events_pass_or_ticketing"].sum(),
    "start_time_below5min_for_three_vehicle_events_pass_or_ticketing": df_summary_area["start_time_below5min_for_three_vehicle_events_pass_or_ticketing"].sum(),
    "start_time_delay_over5min_for_three_vehicle_events_pass_or_ticketing": df_summary_area["start_time_delay_over5min_for_three_vehicle_events_pass_or_ticketing"].sum(),
    "start_time_below10min_for_three_vehicle_events_pass_or_ticketing": df_summary_area["start_time_below10min_for_three_vehicle_events_pass_or_ticketing"].sum(),
    "start_time_delay_over10min_for_three_vehicle_events_pass_or_ticketing": df_summary_area["start_time_delay_over10min_for_three_vehicle_events_pass_or_ticketing"].sum(),
    "dif_extension_observedscheduled_for_three_vehicle_events_pass_or_ticketing_km": df_summary_area["dif_extension_observedscheduled_for_three_vehicle_events_pass_or_ticketing_km"].sum(),
    "dif_median_speed_observedscheduled_for_three_vehicle_events_pass_or_ticketing_kmh": df_summary_area["dif_median_speed_observedscheduled_for_three_vehicle_events_pass_or_ticketing_kmh"].median(),
    "pct_three_events_pass_or_ticketing_from_total_trips": df_summary_area["pct_three_events_pass_or_ticketing_from_total_trips"].mean(),
    "pct_three_events_pass_or_ticketing_oposite_from_total_trips": df_summary_area["pct_three_events_pass_or_ticketing_oposite_from_total_trips"].mean(),
    "pct_expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing_from_three_events_pass_or_ticketing": df_summary_area["pct_expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing_from_three_events_pass_or_ticketing"].mean(),
    "pct_start_time_below5min_for_three_vehicle_events_pass_or_ticketing_from_three_events_pass_or_ticketing": df_summary_area["pct_start_time_below5min_for_three_vehicle_events_pass_or_ticketing_from_three_events_pass_or_ticketing"].mean(),
    "pct_expected_start_time_fail_or_na_from_three_events_pass_or_ticketing": df_summary_area["pct_expected_start_time_fail_or_na_from_three_events_pass_or_ticketing"].mean(),
    "pct_three_events_pass_with_no_ticketing_from_three_events_pass_or_ticketing": df_summary_area["pct_three_events_pass_with_no_ticketing_from_three_events_pass_or_ticketing"].mean(),
    "average_total_trips_per_day": df_summary_area["average_total_trips_per_day"].mean(),
    "average_three_events_pass_or_ticketing_per_day": df_summary_area["average_three_events_pass_or_ticketing_per_day"].mean(),
    "average_expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing_per_day": df_summary_area["average_expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing_per_day"].mean(),
    "average_start_time_below5min_for_three_vehicle_events_pass_or_ticketing_per_day": df_summary_area["average_start_time_below5min_for_three_vehicle_events_pass_or_ticketing_per_day"].mean(),
    "average_extension_observed_for_three_vehicle_events_pass_or_ticketing_km_per_day": df_summary_area["average_extension_observed_for_three_vehicle_events_pass_or_ticketing_km_per_day"].mean(),
    "average_extension_observed_for_three_vehicle_events_pass_or_ticketing_km_per_three_events_pass_or_ticketing": df_summary_area["average_extension_observed_for_three_vehicle_events_pass_or_ticketing_km_per_three_events_pass_or_ticketing"].mean(),
    "dif_three_events_pass_or_ticketing_from_total_trips": df_summary_area["dif_three_events_pass_or_ticketing_from_total_trips"].sum(),
    "average_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km_per_day": df_summary_area["average_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km_per_day"].mean(),
    "average_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km_per_three_events_pass_or_ticketing": df_summary_area["average_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km_per_three_events_pass_or_ticketing"].mean(),
    "dif_extension_scheduled_extension_sheduled_of_three_vehicle_events_pass_or_ticketing_km": df_summary_area["dif_extension_scheduled_extension_sheduled_of_three_vehicle_events_pass_or_ticketing_km"].sum(),
    "pct_three_events_pass_or_ticketing_from_total_trips": df_summary_area["pct_three_events_pass_or_ticketing_from_total_trips"].mean(),
    "passengers_observed_per_three_events_pass_or_ticketing": df_summary_area["passengers_observed_per_three_events_pass_or_ticketing"].mean(),
    "passengers_observed_per_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km": df_summary_area["passengers_observed_per_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km"].mean(),

}

In [ ]:
legend_row = {
    "agency_id": "Legenda",
    "operational_date":"Dias operacionais",
    "passengers_observed":"Passageiros transportados",
    "total_trips":"Viagens planeadas",
    "three_vehicle_events_pass":"Viagens que passaram no teste dos 3 momentos",
    "three_vehicle_events_fail":"Viagens que falharam no teste dos 3 momentos",
    "three_events_pass_or_ticketing":"Viagens planeadas executadas (passaram no teste dos 3 momentos ou têm validações)",
    "three_events_fail_with_ticketing":"Viagens planeadas que falharam o teste dos 3 momentos e têm validações",
    "three_events_pass_with_no_ticketing":"Viagens planeadas executadas sem passageiros",
    "ticketing":"Viagens planeadas com bilhética",
    "extension_observed_km":"Km percorridos (odómetro)",
    "extension_scheduled_km":"Km planeados",
    "extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km":"Km planeados das viagens executadas",
    "extension_observed_for_three_vehicle_events_pass_or_ticketing_km":"Km percorridos (odómetro) das viagens executadas",
    "median_extension_dif_observedscheduled_for_three_vehicle_events_pass_or_ticketing_km":"Mediana das diferenças entre Km percorridos (odómetro)  e Km planeados para as viagens executadas",
    "time_duration_observed_for_three_vehicle_events_pass_or_ticketing_h":"Duração observada (start, endtime observed) das viagens executadas (nº horas)",
    "median_time_duration_observed_for_three_vehicle_events_pass_or_ticketing_min":"Duração mediana observada por viagem executada (min)",
    "median_speed_observed_for_three_vehicle_events_pass_or_ticketing_kmh":"Velocidade mediana observada (start, endtime, extension observed) das viagens executadas",
    "median_speed_scheduled_for_three_vehicle_events_pass_or_ticketing_kmh":"Velocidade mediana planeada (start, endtime, extension scheduled) das viagens executadas",
    "expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing":"Viagens executadas iniciadas dentro da hora prevista [-1min; +5min[",
    "expected_start_time_fail_for_three_vehicle_events_pass_or_ticketing":"Viagens executadas iniciadas fora da hora prevista ]-inf;-1min[ e [+5min; +inf[",
    "start_time_below5min_for_three_vehicle_events_pass_or_ticketing":"Viagens executadas iniciadas até 5min depois da hora prevista [-inf; +5min[",
    "start_time_delay_over5min_for_three_vehicle_events_pass_or_ticketing":"Viagens executadas iniciadas mais de 5min depois da hora prevista[+5min; +inf[",
    "start_time_below10min_for_three_vehicle_events_pass_or_ticketing":"Viagens executadas iniciadas até 10min depois da hora prevista [-inf; +10min[",
    "start_time_delay_over10min_for_three_vehicle_events_pass_or_ticketing":"Viagens executadas iniciadas mais de 10min depois da hora prevista[+10min; +inf[",
    "dif_extension_observedscheduled_for_three_vehicle_events_pass_or_ticketing_km":"Diferença entre Km percorridos (odómetro)  e Km planeados para as viagens executadas",
    "dif_median_speed_observedscheduled_for_three_vehicle_events_pass_or_ticketing_kmh":"Diferença entre a mediana de velocidade observada e a mediana da velocidade planeada das viagens executadas",
    "pct_three_events_pass_or_ticketing_from_total_trips":"Percentagem de viagens planeadas executadas",
    "pct_three_events_pass_or_ticketing_oposite_from_total_trips":"Percentagem de viagens planeadas não executadas",
    "pct_expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing_from_three_events_pass_or_ticketing":"Percentagem de viagens executadas iniciadas dentro da hora prevista [-1min; +5min[",
    "pct_start_time_below5min_for_three_vehicle_events_pass_or_ticketing_from_three_events_pass_or_ticketing":"Percentagem de viagens executadas iniciadas até 5min depois da hora prevista [-inf; +5min[",
    "pct_expected_start_time_fail_or_na_from_three_events_pass_or_ticketing":"Percentagem de viagens executadas iniciadas fora da hora prevista ]-inf;-1min[ e [+5min; +inf[ ou sem hora de início observada",
    "pct_three_events_pass_with_no_ticketing_from_three_events_pass_or_ticketing":"Percentagem de viagens executadas vazias",
    "average_total_trips_per_day":"Média de viagens planeadas por dia",
    "average_three_events_pass_or_ticketing_per_day":"Média de viagens executadas por dia",
    "average_expected_start_time_pass_for_three_vehicle_events_pass_or_ticketing_per_day":"Média de viagens executadas iniciadas dentro da hora prevista [-1min; +5min[ por dia",
    "average_start_time_below5min_for_three_vehicle_events_pass_or_ticketing_per_day":"Média de viagens executadas iniciadas até 5min depois da hora prevista [-inf; +5min[ por dia",
    "average_extension_observed_for_three_vehicle_events_pass_or_ticketing_km_per_day":"Média de Km percorridos (odómetro) das viagens executadas por dia (Km / dia)",
    "average_extension_observed_for_three_vehicle_events_pass_or_ticketing_km_per_three_events_pass_or_ticketing":"Média de Km percorridos (odómetro) das viagens executadas por viagem executada (Km / viagem)",
    "dif_three_events_pass_or_ticketing_from_total_trips":"Viagens planeadas não executadas",
    "average_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km_per_day":"Média de Km planeados das viagens executadas por dia",
    "average_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km_per_three_events_pass_or_ticketing":"Média de Km planeados das viagens executadas por viagem executada",
    "dif_extension_scheduled_extension_sheduled_of_three_vehicle_events_pass_or_ticketing_km": "Diferença entre Km planeados para as viagens executadas e os Km planeados",
    "pct_three_events_pass_or_ticketing_from_total_trips": "Percentagem de viagens planeadas executadas",
    "passengers_observed_per_three_events_pass_or_ticketing": "Passageiros transportados por viagem executada",
    "passengers_observed_per_extension_scheduled_for_three_vehicle_events_pass_or_ticketing_km": "Passageiros por Km planeado das viagens executadas",
}

In [ ]:
import numpy as np
import pandas as pd

# Create the AML row


# Ensure all other columns exist and are NaN
for col in df_summary_area.columns:
    aml_row.setdefault(col, np.nan)
    legend_row.setdefault(col, np.nan)


# Append to dataframe
df_summary_area = pd.concat(
    [df_summary_area, pd.DataFrame([aml_row, legend_row])],
    ignore_index=True
)

# move Legenda to the top
df_summary_area = pd.concat([df_summary_area.tail(1), df_summary_area.iloc[:-1]], ignore_index=True)


In [ ]:
df_summary_area

In [ ]:
df_summary_area.T

In [ ]:
df_summary_area.iloc[0, :].to_list()

order = [
    0,
    1, 2,
    3, 4, 5, 6,
    39,
    7, 8, 9,
    33, 34, 35, 36,
    27, 28,
    43,
    32,
    19, 20, 21, 22, 23, 24,
    29, 30, 31,
    17, 18,
    10, 11, 12, 13, 14,
    25,
    37, 38,
    40, 41, 42,
    15, 16,
    44, 45
]

df_summary_area = df_summary_area.iloc[order, :]



In [ ]:
df_summary_area.head(3)

In [ ]:
#save to CSV
df_summary_area.T.to_excel(os.path.join(OUTPUT_FILES_FOLDER, "summary_areas_performance_2026-01.xlsx"), engine='xlsxwriter',
     index=True
 )
